### Step 1: Install necesscary packages

In [1]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

You should consider upgrading via the 'C:\Users\Jeff\Oliver's Stuff\Coding\NanoGPT-Math\venv\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\Jeff\Oliver's Stuff\Coding\NanoGPT-Math\venv\Scripts\python.exe -m pip install --upgrade pip' command.


### Step 2: Package imports and configuration

In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [4]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [5]:
# Load data from ../data/pos_neg_pairs.json
with open("../data/pos_neg_pairs.json", "r") as f:
    lines = json.load(f)
print(f"Loaded {len(lines)} training pairs")

Loaded 100000 training pairs


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [6]:
# Build optimizer and scheduler using AdamW
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.95)
print(f"Optimizer: AdamW with lr={base_lr}")
print(f"Scheduler: StepLR with gamma=0.95")

Optimizer: AdamW with lr=0.0001
Scheduler: StepLR with gamma=0.95


### Step 7: Begin training (**students are required to complete this part!**)

In [8]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...
        # neg_logprob
        # pos_logprob
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        # ...
        ###########################################################

        # move batch to device
        neg_tensor = neg_tensor.to(device)
        pos_tensor = pos_tensor.to(device)

        optimizer.zero_grad(set_to_none=True)

        # log-probs (helpers provided in template)
        neg_logprob = compute_logprob(neg_tensor)   # [B]
        pos_logprob = compute_logprob(pos_tensor)   # [B]
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # progress
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         lr=optimizer.param_groups[0]['lr'])

    ckpt_path = f"../dpo/dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)

1562it [1:02:06,  2.39s/it, loss=0.1139, lr=2.03e-42]
1562it [1:01:15,  2.35s/it, loss=0.1144, lr=3.26e-77]
1562it [1:01:14,  2.35s/it, loss=0.1147, lr=5.21e-112]
1562it [1:01:14,  2.35s/it, loss=0.1144, lr=8.34e-147]
1562it [1:01:14,  2.35s/it, loss=0.1164, lr=1.33e-181]


### Step 8: Begin testing (**students are required to complete this part!**)

In [9]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
        x = torch.tensor(
            [pad_or_truncate(prompt_ids, max_length)],
            dtype=torch.long, device=device
        )
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)

        # --- FIX: flatten to a 1D list of ints ---
        out_ids = y[0].detach().view(-1).tolist()      # 1D
        # (optional) ensure plain ints if your tensor dtype/device is funky
        out_ids = [int(t) for t in out_ids]

        out = decode(out_ids)

        print(f"Q: {prompt}")
        print(f"A: {out[len(prompt):].strip().splitlines()[0] if len(out) > len(prompt) else out.strip()}")


Q: 17+19=?
A: Whhhehererere aanss is do you feWhamedo? at's your favorite fooWhododo? I like piza
Q: 3*17=?
A: Whheatherhere ansere is o aare is in? becainuauseeng to rain? Yes,seauake uala
Q: 72/4=?
A: Whhhererherhe aat's ime is 3 ayou eals me? Yes, abecause uals 7Yeuala
Q: 72-x=34,x=?
A: Whhheansererer’hose o at's you obbecald you lin ayou spets? Yes, I pe pe ay footbal
Q: x*11=44,x=?
A: Whhheerhererhere at ise is taao in? It's becaus onr 7 becauals 5121 eals erk e 8 e e e e ala
Q: 3*17=?
A: Whhat's t’hos yoDher o o o o o you obbbbalike aing pe ainting
Q: 72/4=?
A: WhhheWheaerwhat's o is yo o your obe ocWheayour favorite food? I like piza
Q: 72-x=34,x=?
A: WhhhehaaaatsserWherhe aat de t's you obe yoke you s? I like paintinggg to rain? Yes, take an umbela
